# 01 · Train RGB → thermal (Pix2Pix, UAV-weighted)

Code lives on GitHub and is cloned in below; this notebook only orchestrates.
Edit code in the repo and re-run the setup cell — never paste code into cells,
or the run stops being reproducible.

**Settings → Accelerator: GPU · Internet: ON**

### The 12-hour problem

A Kaggle session is killed at 12 hours with no warning. Training checkpoints
after every epoch, so a run simply continues in the next session — see the last
section for how.

In [ ]:
# --- Pull the project code from GitHub -------------------------------------
# Requires "Internet" to be ON in the notebook settings panel on the right.
REPO_URL = "https://github.com/Astrq23/rgb-2-thermal.git"
BRANCH   = "main"

import os, subprocess, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/rgb-2-thermal")

if REPO_DIR.exists():
    # Re-running the notebook: fast-forward instead of re-cloning.
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True
    )

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))

print("repo:", REPO_DIR)
print("commit:", subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True).stdout.strip())

In [ ]:
# Editable install so `import rgb2thermal` works everywhere, including inside
# DataLoader worker processes. --no-deps keeps Kaggle's preinstalled torch.
!pip install -e . --no-deps -q

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 1 · Manifest

Skip to section 2 if you already ran notebook 00 in this session.

In [ ]:
import subprocess
from pathlib import Path

MANIFEST = Path("/kaggle/working/manifest.csv")

if MANIFEST.exists():
    print(f"reusing {MANIFEST}")
else:
    subprocess.run(
        ["python", "scripts/build_manifest.py", "--out", str(MANIFEST)], check=True
    )

## 2 · Smoke run (~2 minutes)

Fifty steps at a small resolution. It exercises the manifest, the DataLoader,
both networks, the loss, AMP, checkpointing and image dumping.

Two minutes here has repeatedly been worth more than the ten hours it protects:
a config typo that surfaces at hour nine costs a whole session.

In [ ]:
!python scripts/train.py \
    --config configs/smoke.yaml \
    --manifest /kaggle/working/manifest.csv

In [ ]:
from pathlib import Path
from IPython.display import Image as ShowImage, display

samples = sorted(Path("/kaggle/working/outputs/smoke/samples").glob("*.png"))
if samples:
    display(ShowImage(filename=str(samples[-1]), width=760))
else:
    print("No sample grid was written — check the smoke run output above.")

The output will be noise — fifty steps is nothing. What matters is that the grid
rendered at all, with three columns and the correct number of rows.

## 3 · Full training

Defaults (`configs/pix2pix_uav.yaml`): 40 epochs × 20,000 samples at 256×256,
batch 16, mixed precision. On a P100 that is roughly 8–10 hours — deliberately
inside one session.

The dataset mixture is weighted, not concatenated: DroneVehicle 0.60,
LLVIP 0.30, FLIR 0.10. UAV imagery stays dominant no matter how the download
sizes compare.

**Adjust before starting if needed:**

| Situation | Change |
|---|---|
| T4 ×2 instead of P100 | `--set train.batch_size=24 data.num_workers=4` |
| Out of memory | `--set train.batch_size=8` |
| Want a first result quickly | `--set train.epochs=10` |
| FLIR failed its alignment check | set `enabled: false` in its YAML, then rebuild the manifest |

In [ ]:
!python scripts/train.py \
    --config configs/pix2pix_uav.yaml \
    --manifest /kaggle/working/manifest.csv \
    --resume auto

## 4 · Training curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics = pd.read_csv("/kaggle/working/outputs/pix2pix_uav/metrics.csv")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(metrics["epoch"], metrics["train_g_l1"], label="train")
if "val_l1" in metrics:
    axes[0].plot(metrics["epoch"], metrics["val_l1"], label="val")
axes[0].set_title("L1 reconstruction"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(metrics["epoch"], metrics["train_g_gan"], label="generator")
axes[1].plot(metrics["epoch"], metrics["train_d_total"], label="discriminator")
axes[1].set_title("Adversarial"); axes[1].set_xlabel("epoch"); axes[1].legend()

axes[2].plot(metrics["epoch"], metrics["lr"])
axes[2].set_title("Learning rate"); axes[2].set_xlabel("epoch")

for axis in axes:
    axis.grid(alpha=0.3)
plt.tight_layout(); plt.show()

metrics.tail(10)

**What healthy training looks like**

- `train_g_l1` falls steadily and `val_l1` tracks it. A growing gap is overfitting.
- The adversarial losses oscillate around a rough equilibrium. That is normal.
- `train_d_total` collapsing to ~0 means the discriminator won; the generator
  stops receiving useful gradient. Lower `train.lr`, or raise `train.lambda_l1`.
- `train_g_l1` flat from the very start usually means the data is wrong, not the
  model — go back to the alignment check.

## 5 · Generated samples

In [ ]:
from pathlib import Path
from IPython.display import Image as ShowImage, display

samples = sorted(Path("/kaggle/working/outputs/pix2pix_uav/samples").glob("val_epoch*.png"))
for path in samples[-3:]:
    print(path.name)
    display(ShowImage(filename=str(path), width=760))

Judge these by **thermal plausibility**, not by pixel match: vehicles and people
should be brighter than road and vegetation, engine bays hotter than bodywork,
buildings retaining heat after dark. Getting that ordering right matters more
than matching any individual frame.

## 6 · Evaluate

In [ ]:
!python scripts/evaluate.py \
    --checkpoint /kaggle/working/outputs/pix2pix_uav/checkpoints/best.pt \
    --config configs/pix2pix_uav.yaml \
    --manifest /kaggle/working/manifest.csv \
    --split test

Metrics are reported **per source dataset** on purpose. A single average would
hide the thing that matters here — whether the UAV domain improved, or whether a
gain came from the easier ground-level FLIR frames.

- **PSNR / SSIM** — fidelity to that exact ground-truth frame.
- **LPIPS** — perceptual distance. Usually the most informative of the three.
- **FID** — distributional realism across the whole set.
- **Unpaired FID vs HIT-UAV** — the one metric that asks whether the output looks
  like *real high-altitude UAV thermal* rather than merely matching one frame.

## 7 · Continuing in the next session

Kaggle kills the session at 12 hours. To carry on:

1. **Save Version** → *Save & Run All*, and wait for it to finish.
2. In a new notebook: **Add data → Notebook Output** → select this run.
3. Copy the checkpoint across and resume:

```python
!mkdir -p /kaggle/working/outputs/pix2pix_uav/checkpoints
!cp /kaggle/input/<your-notebook-output>/outputs/pix2pix_uav/checkpoints/last.pt \
    /kaggle/working/outputs/pix2pix_uav/checkpoints/

!python scripts/train.py --config configs/pix2pix_uav.yaml \
    --manifest /kaggle/working/manifest.csv --resume auto
```

The checkpoint carries both networks, both optimisers, both schedulers, the AMP
scaler, the epoch counter and the RNG state, so the loss curve continues without
a visible seam.

Also download `checkpoints/best.pt` before the session ends — Kaggle's working
directory does not survive it.